# 01 · Concurrency: Threads, Processes & `concurrent.futures`

**Module 13 – Concurrency** · Estimated time: 60 min

> Expands 📘 *Multithreading* (one example). Not in the 📗 bootcamp.

### 🎯 What you will learn
- Concurrency vs parallelism; I/O-bound vs CPU-bound work
- The **GIL** and what it means (and how 3.13/3.14 changed things)
- `threading`: starting/joining threads, race conditions, `Lock`, `Queue`
- `multiprocessing`: true parallelism for CPU-bound work
- ✅ `concurrent.futures`: `ThreadPoolExecutor` / `ProcessPoolExecutor` (the easy, modern API)
- Choosing the right tool

## 1. Key ideas

- **Concurrency**: dealing with many things at once (switching between tasks while waiting)
- **Parallelism**: doing many things *at the same instant* (multiple CPU cores)

| Work type | Bottleneck | Examples | Best tool |
|---|---|---|---|
| **I/O-bound** | waiting for network/disk | downloading pages, API calls, DB queries | threads or `asyncio` |
| **CPU-bound** | computation | image processing, number crunching | processes (or free-threaded Python) |

### The GIL (Global Interpreter Lock)
In standard CPython, the GIL allows only **one thread to execute Python bytecode at a time**. So:
- Threads **do** speed up I/O-bound work (the GIL is released while waiting)
- Threads **don't** speed up CPU-bound pure-Python work → use processes

> 🆕 Python 3.13 introduced an experimental **free-threaded** build without the GIL; in 3.14 it became officially supported (PEP 779), and the 3.15 macOS installer installs it by default. See lesson 08.

## 2. `threading` basics (📘)

In [1]:
import threading
import time

def worker(num):
    print(f"Thread {num}: Starting")
    time.sleep(1)                 # simulate I/O (e.g. a network request)
    print(f"Thread {num}: Finishing")

start = time.perf_counter()
threads = []
for i in range(3):
    thread = threading.Thread(target=worker, args=(i,))
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()                 # wait for all threads to finish

print(f"All threads completed in {time.perf_counter() - start:.2f}s (not 3s!)")

Thread 0: StartingThread 1: Starting

Thread 2: Starting


Thread 0: FinishingThread 2: Finishing

Thread 1: Finishing
All threads completed in 1.01s (not 3s!)


Three 1-second waits finished in about **1 second total**, because the threads waited in parallel.

Notice the output lines may be jumbled (e.g. two messages on one line). Threads run in an unpredictable order, and their prints can interleave. That's normal, and it's a first hint of why shared resources need care.

### Sequential vs threaded "downloads"

In [2]:
def fake_download(url):
    time.sleep(0.5)               # pretend network latency
    return f"{url}: done"

urls = [f"https://example.com/page{i}" for i in range(6)]

start = time.perf_counter()
results = [fake_download(u) for u in urls]
print(f"sequential: {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
threads = [threading.Thread(target=fake_download, args=(u,)) for u in urls]
for t in threads: t.start()
for t in threads: t.join()
print(f"threaded:   {time.perf_counter() - start:.2f}s")

sequential: 3.03s


threaded:   0.51s


## 3. Race conditions & `Lock`

When threads modify **shared data**, operations can interleave and corrupt it:

In [3]:
counter = 0

def unsafe_increment(n):
    global counter
    for _ in range(n):
        current = counter          # read
        time.sleep(0)              # yield to other threads (makes the race visible)
        counter = current + 1      # write: may overwrite another thread's update!

threads = [threading.Thread(target=unsafe_increment, args=(1000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("expected 4000, got", counter)

expected 4000, got 1000


In [4]:
counter = 0
lock = threading.Lock()

def safe_increment(n):
    global counter
    for _ in range(n):
        with lock:                 # only one thread at a time inside this block
            current = counter
            time.sleep(0)
            counter = current + 1

threads = [threading.Thread(target=safe_increment, args=(1000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("expected 4000, got", counter)

expected 4000, got 4000


## 4. Producer–consumer with `queue.Queue` (thread-safe)

In [5]:
import queue

tasks = queue.Queue()
results = []

def consumer(name):
    while True:
        item = tasks.get()
        if item is None:           # sentinel: time to stop
            tasks.task_done()
            break
        results.append(f"{name} processed {item}")
        tasks.task_done()

workers = [threading.Thread(target=consumer, args=(f"W{i}",)) for i in range(2)]
for w in workers: w.start()
for item in range(6):
    tasks.put(item)                # producer
for _ in workers:
    tasks.put(None)
tasks.join()
print(len(results), results[:3])

6 ['W0 processed 0', 'W0 processed 1', 'W0 processed 2']


## 5. ✅ `concurrent.futures`: the high-level API

Instead of managing threads by hand, submit work to a **pool**:

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as executor:
    results = list(executor.map(fake_download, urls))     # results keep input order
print(results[:2], f"{time.perf_counter() - start:.2f}s")

['https://example.com/page0: done', 'https://example.com/page1: done'] 0.51s


In [7]:
# submit() returns Future objects; as_completed yields them as they finish
import random

def task(n):
    time.sleep(random.uniform(0.1, 0.4))
    if n == 3:
        raise ValueError("task 3 failed")
    return n * n

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(task, n): n for n in range(6)}
    for fut in as_completed(futures):
        n = futures[fut]
        try:
            print(f"task {n} → {fut.result()}")
        except ValueError as e:
            print(f"task {n} raised: {e}")

task 0 → 0
task 2 → 4


task 3 raised: task 3 failed
task 5 → 25


task 1 → 1


task 4 → 16


### Real HTTP requests in parallel

In [8]:
import urllib.request

def fetch_status(url):
    with urllib.request.urlopen(url, timeout=10) as resp:
        return url, resp.status

sites = ["https://www.python.org", "https://docs.python.org/3/", "https://pypi.org"]
with ThreadPoolExecutor() as ex:
    for url, status in ex.map(fetch_status, sites):
        print(status, url)

200 https://www.python.org
200 https://docs.python.org/3/
200 https://pypi.org


## 6. CPU-bound work: processes

Threads don't help CPU-bound Python code (GIL). **Processes** each have their own interpreter and GIL, so they run on separate cores.

> ⚠️ With multiprocessing, worker functions must be **importable** (defined in a `.py` file), especially on Windows and macOS, which start processes with *spawn*. In scripts, always protect the entry point with `if __name__ == "__main__":`.

In [9]:
%%writefile cpu_tasks.py
def count_primes(limit):
    """Deliberately CPU-heavy: count primes below limit."""
    count = 0
    for n in range(2, limit):
        if all(n % d for d in range(2, int(n ** 0.5) + 1)):
            count += 1
    return count

Writing cpu_tasks.py


In [10]:
from concurrent.futures import ProcessPoolExecutor
from cpu_tasks import count_primes
import os

jobs = [200_000] * 4
start = time.perf_counter()
seq = [count_primes(j) for j in jobs]
t_seq = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    thr = list(ex.map(count_primes, jobs))
t_thr = time.perf_counter() - start

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as ex:
    par = list(ex.map(count_primes, jobs))
t_par = time.perf_counter() - start

print(f"CPU cores: {os.cpu_count()}")
print(f"sequential: {t_seq:.2f}s | threads: {t_thr:.2f}s | processes: {t_par:.2f}s")
print(seq == thr == par)

CPU cores: 8
sequential: 0.63s | threads: 0.62s | processes: 0.24s
True


Threads gave no speed-up for CPU work (the GIL), while processes used multiple cores.

### The `multiprocessing` module directly

```python
from multiprocessing import Pool
from cpu_tasks import count_primes

if __name__ == "__main__":
    with Pool(processes=4) as pool:
        print(pool.map(count_primes, [200_000] * 4))
```
Processes don't share memory: arguments and results are **pickled** and copied between processes, so keep them small.

## 7. Choosing the right tool

| Situation | Use |
|---|---|
| Many network/disk waits, simple code | `ThreadPoolExecutor` |
| Thousands of concurrent connections | `asyncio` (lessons 05–07) |
| CPU-heavy pure Python | `ProcessPoolExecutor` / `multiprocessing` |
| CPU-heavy, need shared memory, modern Python | free-threaded build or subinterpreters (lesson 08) |
| Numeric arrays | NumPy (releases the GIL internally) |

> ➡️ This lesson is the overview. Deep dives: **02** threading & synchronisation primitives · **03** multiprocessing · **04** concurrent.futures · **05–07** asyncio · **08** free-threading & subinterpreters.

## 📝 Summary
- Threads speed up I/O-bound work; the GIL blocks CPU parallelism in standard builds.
- Protect shared data with `Lock`; communicate safely with `queue.Queue`.
- ✅ `concurrent.futures` executors: `map`, `submit`, `as_completed`.
- Processes give real parallelism for CPU-bound work; put workers in importable modules.

## 🏋️ Practice
1. Use `ThreadPoolExecutor` to "download" 10 fake URLs (sleep 0.3s each) and print the total time.
2. Write a thread-safe `BankAccount` with a `Lock` and run 5 threads depositing ₹100 a thousand times each. Check the final balance.
3. Use `ProcessPoolExecutor` to compute `sum(i*i for i in range(n))` for several large `n`.

<details><summary>Show solutions</summary>

```python
import time, threading
from concurrent.futures import ThreadPoolExecutor

def fake(u):
    time.sleep(0.3)
    return u
t = time.perf_counter()
with ThreadPoolExecutor(10) as ex:
    list(ex.map(fake, range(10)))
print(time.perf_counter() - t)

class BankAccount:
    def __init__(self):
        self.balance = 0
        self._lock = threading.Lock()
    def deposit(self, amount):
        with self._lock:
            self.balance += amount
acct = BankAccount()
def run():
    for _ in range(1000):
        acct.deposit(100)
ts = [threading.Thread(target=run) for _ in range(5)]
for x in ts: x.start()
for x in ts: x.join()
print(acct.balance)     # 500000

# squares.py:  def sum_sq(n): return sum(i * i for i in range(n))
# from concurrent.futures import ProcessPoolExecutor
# with ProcessPoolExecutor() as ex: print(list(ex.map(sum_sq, [10**6, 2*10**6, 3*10**6])))
```
</details>